In [1]:
import pandas as pd
print("pandas is working")

pandas is working


In [6]:
import pandas as pd
df = pd.read_csv('rune_hunter_equipment.csv')
print(df.shape)
print(df.columns.tolist())

(64, 6)
['Category', 'Item', 'Damage / Rating', 'Slots', 'Treasure', 'Notes']


this creates a dataframe from a csv named in parentheses. print(df.shape) prints the number of rows and columns. print(df.columns.tolist()) prints the top row/header of each column.

In [7]:
df.head(10)

,Category,Item,Damage / Rating,Slots,Treasure,Notes
0,WEAPONS,Unarmed,d4,-,-,Natural attack; Chimera with natural weapons u...
1,WEAPONS,Dagger / knife,d4,1,<1,Concealable; thrown uses AGI
2,WEAPONS,Short sword / hand axe,d4,1,<1,Thrown axe uses STR
3,WEAPONS,Sword / saber / mace,d6,1,<1,-
4,WEAPONS,Spear,d6,1,<1,Brace action: +2 ATK vs mounted charge
5,WEAPONS,Staff,d6,1,<1,Two-handed
6,WEAPONS,Greatsword / greataxe,d8,1,1,Two-handed
7,WEAPONS,War hammer / maul,d8,1,1,Two-handed
8,WEAPONS,Bow,d6,1,1,Ranged; needs quiver (1 slot)
9,WEAPONS,Crossbow,d8,1,1,Ranged; reload costs action (fires every other...


This prints the top (n) rows of the current dataframe (which we set earlier with df = pd.read.csv).

In [8]:
df['Category'].value_counts()

Category
GEAR        26
WEAPONS     14
SERVICES    11
MOUNTS       7
ARMOR        6
Name: count, dtype: int64

This looks at a specific column by name, and then lists the count of each value present in that column.

The Slots column contains mixed text and numeric values, so `.mean()` falls on the raw column. See next cell for the cleaning approach.

In [9]:
df.groupby('Category')['Slots'].mean().round(2)

TypeError: dtype 'str' does not support operation 'mean'

This errored because I tried to use 'mean' on a column that had non-numeric characters.

In [10]:
df['Slots'].value_counts()

Slots
1         30
-         21
<1         6
1 body     3
2 body     2
1 hand     1
 1         1
Name: count, dtype: int64

Similar to earlier, I used value_counts to get the unique values for 'Slots' to assess what caused the previous error.

Slots column cleaned: `-` and `<1` treated as 0 (negligible encumbrance). Location qualifiers (`1 body`, `1 hand`) treated as 1 slot.

In [11]:
df['Slots_numeric'] = pd.to_numeric(df['Slots'].str.extract(r'(\d+)')[0], errors='coerce').fillna(0)
df.groupby('Category')['Slots_numeric'].mean().round(2)

Category
ARMOR       1.33
GEAR        0.92
MOUNTS      0.00
SERVICES    0.00
WEAPONS     0.93
Name: Slots_numeric, dtype: float64

This looks at Slots_numeric, Uses str.extract(r'(\d+)') to pull the first sequence of digits from each value in the Slots column, then pd.to_numeric to convert those extracted strings into actual numbers. errors='coerce' replaces anything that can't convert with NaN instead of crashing. fillna(0) replaces those NaN values with 0, treating negligible-slot items as zero. Then groups by Category and calculates the mean slot cost per group, rounded to 2 decimal places.